In [ ]:
# Referenced/Used material from AI healthcare class
# Referenced/Used previous work completed by me for AI healthcare class
from google.cloud import bigquery
from google.colab import auth, userdata
from sklearn.model_selection import train_test_split
from pathlib import Path

auth.authenticate_user()

# Referenced/Used: https://stackoverflow.com/questions/66631333/how-do-i-set-environment-variables-in-google-colab
project_id = userdata.get("PROJECT_ID")
# End of: Referenced/Used: https://stackoverflow.com/questions/66631333/how-do-i-set-environment-variables-in-google-colab

client = bigquery.Client(project=project_id)

In [26]:
# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (SQL CASE query help)
# Referenced/Used: https://www.programiz.com/sql/like-operator
radiology_query = """
SELECT
radiology.subject_id AS subject_id,
radiology.hadm_id AS hadm_id,
radiology.text AS notes,
d_icd_diagnoses.icd_version AS icd_version,
d_icd_diagnoses.icd_code AS icd_code,
d_icd_diagnoses.long_title AS title,
CASE
    WHEN LOWER(d_icd_diagnoses.long_title) LIKE '%malignant%' THEN 1
    ELSE 0
END AS is_malignant_diagnoses
FROM `physionet-data.mimiciv_note.radiology` AS radiology
JOIN `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` AS diagnoses_icd
ON radiology.subject_id = diagnoses_icd.subject_id
AND radiology.hadm_id = diagnoses_icd.hadm_id
JOIN `physionet-data.mimiciv_3_1_hosp.d_icd_diagnoses` AS d_icd_diagnoses
ON d_icd_diagnoses.icd_code = diagnoses_icd.icd_code
AND d_icd_diagnoses.icd_version = diagnoses_icd.icd_version
WHERE (d_icd_diagnoses.icd_version = 9 AND diagnoses_icd.icd_version = 9
AND LOWER(d_icd_diagnoses.long_title) LIKE '%tumor%')
AND (LOWER(d_icd_diagnoses.long_title) LIKE '%malignant%'
OR LOWER(d_icd_diagnoses.long_title) LIKE '%benign%');
"""
# End of: Referenced/Used: https://www.programiz.com/sql/like-operator
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (SQL CASE query help)

# Referenced/Used: https://docs.cloud.google.com/python/docs/reference/bigquery/latest
result = client.query_and_wait(radiology_query)
df = result.to_dataframe()
# End of: Referenced/Used: https://docs.cloud.google.com/python/docs/reference/bigquery/latest

,subject_id,hadm_id,notes,icd_version,icd_code,title,is_malignant_diagnoses
0,15739491,20214886,INTRAOPERATIVE ULTRASOUND OF THE LIVER AND PAN...,9,20901,Malignant carcinoid tumor of the duodenum,1
1,18870530,26912617,ULTRASOUND INTERVENTIONAL PROCEDURE: Ultrasou...,9,20963,Benign carcinoid tumor of the stomach,0
2,18870530,23384804,"EXAM: Chest, frontal and lateral views.\n\nCL...",9,20963,Benign carcinoid tumor of the stomach,0
3,14693603,25034528,INDICATION: ___ male with chest soreness.\n\n...,9,20913,Malignant carcinoid tumor of the ascending colon,1
4,17979567,21271258,EXAMINATION: CHEST (PA AND LAT)\n\nINDICATION...,9,20966,"Benign carcinoid tumor of midgut, not otherwis...",0
...,...,...,...,...,...,...,...
1521,19199806,20672485,"HISTORY: Remote lumbar spinal fusion, now wit...",9,V1091,Personal history of malignant neuroendocrine t...,1
1522,19199806,20672485,INDICATION: Past medical history of right hip...,9,V1091,Personal history of malignant neuroendocrine t...,1
1523,16114976,26279410,"AP CHEST, 9:24 A.M ON ___\n\nHISTORY: Febrile...",9,V1091,Personal history of malignant neuroendocrine t...,1
1524,10670085,25603584,INDICATION: Rule out pulmonary abnormality or...,9,V1091,Personal history of malignant neuroendocrine t...,1


In [ ]:
# Referenced/Used: https://www.geeksforgeeks.org/python/python-pandas-dataframe-dropna/
# Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
df = df.dropna(axis=0, subset=['notes', 'is_malignant_diagnoses'])
df = df[['notes', 'is_malignant_diagnoses']]
x_train, y_train, x_test, y_test = train_test_split(df['notes'], df['is_malignant_diagnoses'], train_size = 0.7, random_state=42)
# End of: Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

train_path = Path("/content/data/train")
validation_path = Path("/content/data/validation")
train_path.mkdir(exist_ok=True, parents=True)
validation_path.mkdir(exist_ok=True, parents=True)


x_train.to_csv(train_path / "features.csv")
x_test.to_csv(train_path / "results.csv")

y_train.to_csv(validation_path / "features.csv")
y_test.to_csv(validation_path / "results.csv")
# End of: Referenced/Used: https://www.geeksforgeeks.org/python/python-pandas-dataframe-dropna/
# End of: Referenced/Used previous work completed by me for AI healthcare class
# End of: Referenced/Used material from AI healthcare class
